# 🔬 SPV GPU: FDTDX (JAX/GPU) + 本家 Harminv CLI 共振モード解析ノートブック

本ノートブックは **仕様書B1（FDTDX + 本家 Harminv CLI への移植）** および **仕様書A（全自動パイプライン・パラメータ自動導出）** に準拠した Google Colab 実行環境です。
各種コードは GitHub リポジトリ [`kyouoku/colab_GPU`](https://github.com/kyouoku/colab_GPU.git) から自動取得します。

### パイプライン構成
1. **STEP 1（広帯域探索）**: 複数点・複数成分のガウシアンパルス双極子源を同時励振し、FDTDX（`jax.lax.scan` チャンク実行）で時間発展を計算 → 本家 `harminv` CLI を並列実行して共振ピークを抽出
2. **STEP 2（モード自動選別）**: 近接周波数クラスタリング（$|\Delta f| < f / (2Q)$）と $Q$ 中央値・最大振幅ランキングにより、上位 `n_modes_select` 個の共振モードを自動選出（`best condition {date}.txt` 自動生成）
3. **STEP 2.5（高解像度精密化）**: 4x4x4 サブピクセル誘電率平滑化＋高解像度グリッド（`res_mode`）で狭帯域励振し、共振周波数 $f$ と $Q$ 値を精密化
4. **STEP 3（モード空間分布出力）**: 確定した各共振モード周波数で Running DFT（`xy (z=0)`, `xy (z=±0.06)`, `xz (y=0)`, `yz (x=0)` 断面）を計算し、電場エネルギー密度 $D_{\text{den}} = \frac{1}{2}\varepsilon |\mathbf{E}|^2$・磁場エネルギー密度 $H_{\text{den}} = \frac{1}{2}|\mathbf{H}|^2$ の断面プロファイル画像を生成

> **💡 ヒント**: Google Colab のメニューから **「ランタイム > ランタイムのタイプを変更 > ハードウェアアクセラレータ: GPU (T4 / L4 / A100)」** を選択してから上から順に実行してください。

In [ ]:
# ==============================================================================
# 1. 環境セットアップ & FDTDX + 本家 Harminv CLI のインストール
# ==============================================================================
import os
import shutil
import subprocess
import sys

print("📦 Installing / verifying FDTDX (JAX) and dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fdtdx", "pandas", "matplotlib"], check=False)

# 本家 Harminv CLI バイナリのインストール (apt -> conda-forge フォールバック)
if shutil.which("harminv") is None:
    print("🔧 Installing upstream harminv CLI via apt-get...")
    subprocess.run(
        "apt-get update -qq && apt-get install -y -qq harminv",
        shell=True,
        capture_output=True,
        text=True,
    )
    if shutil.which("harminv") is None:
        print("⚠️ apt install failed or unavailable; trying conda-forge fallback...")
        subprocess.run("conda install -y -c conda-forge harminv", shell=True, check=False)

harminv_path = shutil.which("harminv")
if harminv_path:
    ver_out = "harminv CLI installed"
    for flag in ("-V", "--version", "-h"):
        res = subprocess.run([harminv_path, flag], capture_output=True, text=True, check=False)
        combined = ((res.stdout or "") + "\n" + (res.stderr or "")).strip()
        if combined:
            first_line = combined.splitlines()[0].strip()
            if "invalid option" not in first_line.lower() and "unrecognized option" not in first_line.lower():
                ver_out = first_line
                break
    print(f"✅ Harminv CLI ready: {ver_out} ({harminv_path})")
else:
    print("⚠️ Harminv CLI binary not found in PATH (will use built-in Matrix-Pencil SVD fallback).")

# GPU & JAX デバイス確認 (BFCアロケータ断片化によるOOMを防止する環境変数を設定)
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
import jax
print(f"✅ JAX version: {jax.__version__} | Backend: {jax.default_backend()} | Devices: {jax.devices()}")
if jax.default_backend() != "gpu":
    print("⚠️ 警告: GPU が検出されていません。Colab の「ランタイム > ランタイムのタイプを変更」で GPU を有効化してください。")

## 2. Google Drive のマウントと GitHub リポジトリ (`https://github.com/kyouoku/colab_GPU.git`) の取得
GitHub リポジトリから最新の `spv_gpu` パッケージをクローン（または `git pull` で更新）し、Google Drive 上の出力先ディレクトリを設定します。

In [ ]:
# ==============================================================================
# 2. Google Drive マウント & GitHub リポジトリ (kyouoku/colab_GPU) の取得
# ==============================================================================
import importlib
import os
import subprocess
import sys

REPO_URL = "https://github.com/kyouoku/colab_GPU.git"
MOUNT_GOOGLE_DRIVE = True  # False にするとローカルの ./spv_results に保存します

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DEFAULT_OUT_DIR = "/content/drive/MyDrive/spv_results"
        print(f"✅ Google Drive mounted. Default output_dir: {DEFAULT_OUT_DIR}")
    except Exception as e:
        DEFAULT_OUT_DIR = os.path.abspath("./spv_results")
        print(f"ℹ️ Running outside Colab or Drive mount skipped ({e}). Using {DEFAULT_OUT_DIR}")
else:
    DEFAULT_OUT_DIR = os.path.abspath("./spv_results")

# リポジトリの取得・更新
if os.path.isdir("spv_gpu") and os.path.isfile("spv_gpu/config.py"):
    REPO_DIR = os.path.abspath(".")
    print(f"📂 Using existing local repository at: {REPO_DIR}")
elif os.path.isdir("../spv_gpu") and os.path.isfile("../spv_gpu/config.py"):
    REPO_DIR = os.path.abspath("..")
    print(f"📂 Using parent repository at: {REPO_DIR}")
else:
    REPO_DIR = "/content/colab_GPU" if os.path.isdir("/content") else os.path.abspath("./colab_GPU")
    if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
        print(f"📥 Cloning {REPO_URL} into {REPO_DIR} ...")
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        print(f"🔄 Pulling latest changes in {REPO_DIR} ...")
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import spv_gpu
import spv_gpu.config
import spv_gpu.geometry.structures
importlib.reload(spv_gpu.config)
importlib.reload(spv_gpu.geometry.structures)
importlib.reload(spv_gpu)
print("✅ spv_gpu package loaded from:", spv_gpu.__file__)

## 3. ユーザ設定パラメータ（`cfg` & `backend`）の編集
**仕様書A §4**・**仕様書B1 §4** および元の `spv/my_geometry.py` / `spv/run_sim.py` と完全互換の構造パラメータを定義します。
- `structure_type`:
  - `6`: Singular Bowtie PhC（ツイスト・ハニカム格子 $N_{\text{ring}}=8$, $\theta=3.5^\circ$ ＋ $x$軸対向ボウタイ特異共振器）
  - `3`: Microdisk（犠牲層ペデスタル付きマイクロディスク共振器）
  - `4`: Rough Microdisk（側壁ラフネス付きマイクロディスク共振器）
  - `5`: Waveguide-coupled Disk（導波路結合マイクロディスク）
  - `0`: 2D PhC Defect Cavity（三角格子フォトニック結晶欠陥共振器、V2検証用）

In [ ]:
# ==============================================================================
# 3. 計算条件の設定 (cfg & backend) と自動導出パラメータの確認
# ==============================================================================
from spv_gpu.config import get_default_config, derive_parameters

cfg, backend = get_default_config()

# --- 構造と探索条件 (spv/run_sim.py & 仕様書A §4 と完全互換) ---
cfg.update(
    structure_type = 6,          # 3:microdisk, 4:rough_disk, 5:wg_disk, 6:singular_bowtie_phc, 0:2D_PhC
    structure      = dict(
        # type 6: Singular Bowtie PhC (spv/my_geometry.py 既定パラメータ)
        n            = 3.4,      # コア屈折率 (eps_r = 11.56)
        a            = 0.45,     # [um] 格子定数 period
        r_ratio      = 0.26,     # 空気孔半径 r / a
        slab_ratio   = 0.50,     # スラブ厚 slab_t / a
        N_ring       = 8,        # ツイストハニカム格子のリング数
        twist_deg    = 3.5,      # 副格子A/Bの相対ツイスト角 [deg]
        core_r_ratio = 0.55,     # 中心ボウタイ領域の外径半径 core_r / a
        gap_ratio    = 0.02,     # 中心ギャップ幅 g / a
        bowtie_angle = 60.0,     # ボウタイ頂角 [deg]
    ),
    f_range        = (0.55, 0.75), # 探索周波数範囲 [a/lambda または um^-1]
    parities       = ["+", "-"],   # z対称パリティ ("+": even/TE-like, "-": odd/TM-like)
    accuracy       = dict(coarse=2, fine=4), # 最小特徴寸法あたりの格子数
    quality        = "standard", # "fast" (テスト用) | "standard" (標準) | "high" (高精度)
    n_modes_select = 3,          # STEP 2 で自動選出する上位モード数 K
    Q_expected     = 1e4,        # 想定Q値 (リングダウン上限時間の算出用)
    run_mode       = "ALL",      # "ALL" (STEP 1->2->2.5->3) | "HARMINV" (STEP 1->2) | "MODE" (STEP 3のみ)
    merge_refine_mode = False,   # True にすると STEP 2.5 と STEP 3 を1回の高解像度FDTDランに統合
    output_dir     = DEFAULT_OUT_DIR,
)

# --- FDTDX / JAX GPU バックエンド設定 ---
backend.update(
    dtype                 = "float32", # "float32" (推奨) | "bfloat16" (高速探索)
    max_gpu_mem_frac      = 0.70,      # VRAM上限の70%を超える場合は自動で res を引き下げ
    x64_refine            = False,     # Q > 1e6 を狙う場合のみ True
    chunk_steps           = 1000,      # lax.scan の1チャンクあたりステップ数
    subpixel_n            = 4,         # STEP 2.5 / 3 の 4x4x4 サブピクセル平滑化
    anisotropic_averaging = False,
)

# 自動導出パラメータのプレビュー表示
derived = derive_parameters(cfg, backend)
print("=== 自動導出された計算パラメータ ===")
for k in [
    "fcen", "df", "feat_min", "res_harminv", "res_mode", "sp_harminv", "sp_mode",
    "grid_harminv", "grid_mode", "parities", "t_src_end",
    "T_MAX_harminv", "T_MAX_refine", "T_MAX_mode",
    "est_mem_harminv_gb", "est_mem_mode_gb", "gpu_mem_gb"
]:
    print(f"  {k:20s}: {derived[k]}")

## 4. 誘電率構造プレビュー（`epchk` 断面確認）
FDTD 時間発展を実行する前に、`spv/my_geometry.py` と同一ジオメトリで生成される 3D 誘電率分布 $\varepsilon(x,y,z)$ の 3 断面（xy, xz, yz）を確認します。

In [ ]:
# ==============================================================================
# 4. 誘電率構造 (epchk) の事前確認とインライン表示
# ==============================================================================
from IPython.display import Image, display
from spv_gpu.geometry.structures import build_structure_permittivity
from spv_gpu.pipeline import plot_epchk_slices

pz_preview = derived["parities"][0]
eps_preview, coords_preview, prefix_str = build_structure_permittivity(
    cfg=cfg,
    sp=derived["sp_mode"],
    res=derived["res_mode"],
    pz=pz_preview,
    eps_averaging=True,
    subpixel_n=backend["subpixel_n"],
)

preview_png = "./epchk_preview.png"
plot_epchk_slices(
    eps_preview,
    coords_preview,
    pz=pz_preview,
    out_png_path=preview_png,
    title=f"Preview: {prefix_str} (res={derived['res_mode']} px/um, pz={pz_preview})",
)
display(Image(filename=preview_png))
del eps_preview
import gc; gc.collect()

## 5. 全自動パイプライン実行（STEP 1 → STEP 2 → STEP 2.5 → STEP 3）
下のセルを実行すると、指定した `run_mode`（既定 `'ALL'`）に従い、広帯域 Harminv 探索からモード自動選別・高解像度精密化・2D 断面モードプロファイル出力までを一気通貫で実行します。
すべての結果（`summary.csv`, `best condition {date}.txt`, `run_log.txt`, 各種 PNG 画像）は Google Drive（`output_dir`）に自動保存されます。

In [ ]:
# ==============================================================================
# 5. 全自動パイプライン (STEP 1 -> STEP 2 -> STEP 2.5 -> STEP 3) の実行
# ==============================================================================
import pandas as pd
from IPython.display import Image, display, Markdown
from spv_gpu.pipeline import run_spv_pipeline

pipeline_result = run_spv_pipeline(cfg=cfg, backend=backend)

print(f"\n📂 出力ディレクトリ: {pipeline_result['output_dir']}")
display(Markdown("### 📊 選出・精密化された共振モード一覧 (summary.csv)"))
display(pipeline_result["summary_df"])

# Harminv 周波数-Q値散布図の表示
for png_path in pipeline_result["harminv_pngs"]:
    display(Markdown(f"#### 📈 Harminv 検出分布: {os.path.basename(png_path)}"))
    display(Image(filename=png_path))

# STEP 3 モード断面プロファイル (D_den, H_den) の表示
for png_path in pipeline_result["mode_profile_pngs"]:
    display(Markdown(f"#### 🌐 共振モード断面プロファイル: {os.path.basename(png_path)}"))
    display(Image(filename=png_path))

## 6. 段階検証・ベンチマークスイート（仕様書B1 §9: V1〜V5）
移植された FDTDX＋本家 Harminv パイプラインの妥当性と GPU 高速化倍率を検証するためのセルです。
- **V1**: `structure_type = 6` のサブピクセル平滑化誘電率 $\varepsilon(x,y,0)$ の相対 L2 誤差検証（$< 2\%$）
- **V4**: 既知の複素減衰正弦波に対する本家 `harminv` CLI の単体精度検証
- **V5**: GPU スループット（Mcells·step/s）および Colab CPU Meep 比の高速化倍率ベンチマーク
- **V2 / V3**: 2D PhC 欠陥共振器（`structure_type = 0`）および Singular Bowtie PhC（`structure_type = 6`）の共振モード抽出テスト

In [ ]:
# ==============================================================================
# 6. 段階検証 (V1, V4, V5 + オプションで V2, V3) の実行
# ==============================================================================
import pandas as pd
from IPython.display import Image, display
from spv_gpu.validate_vs_meep import (
    validate_v1_permittivity,
    validate_v4_harminv_cli,
    validate_v5_speed_benchmark,
    validate_v2_2d_phc,
    validate_v3_type6_bowtie,
)

val_dir = "./spv_validation"
v1_res = validate_v1_permittivity(out_dir=val_dir)
v4_res = validate_v4_harminv_cli()
v5_res = validate_v5_speed_benchmark()

print("✅ V1 (Permittivity L2 Error < 2%):", v1_res)
if "plot_path" in v1_res and os.path.exists(v1_res["plot_path"]):
    display(Image(filename=v1_res["plot_path"]))

print("\n✅ V4 (Harminv CLI Unit Test):", v4_res)
print("\n✅ V5 (GPU Throughput Benchmark):")
display(pd.DataFrame(v5_res["benchmarks"]))

# V2 (2D PhC) & V3 (Type 6 Even/Odd) も続けて検証する場合は True に変更してください
RUN_FULL_V2_V3 = False
if RUN_FULL_V2_V3:
    v2_res = validate_v2_2d_phc(out_dir=val_dir)
    v3_res = validate_v3_type6_bowtie(out_dir=val_dir)
    print("\n✅ V2 (2D PhC Cavity):", v2_res)
    print("\n✅ V3 (Type 6 Bowtie Even/Odd):", v3_res)